# 🛡️ DrugGuard — MobileNetV2 ONNX CNN (Vision) Training

This notebook fine-tunes **MobileNetV2** for classifying illicit drugs and restricted substances (cannabis, cocaine, vape products, tobacco) vs safe packaging/medical items.

## Setup Instructions:
1. Change runtime type to **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
2. Run all cells sequentially.
3. Download the generated `image_model.onnx` and `image_classes.json` files and place them inside `backend/train/` in your local project directory.

### Step 1: Install Dependencies

In [1]:
!pip install -q torch torchvision scikit-learn onnx onnxscript


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 2: Clone Cloned Dataset Repository

In [2]:
!git clone https://github.com/gopeshgopi/Drug-Image-Dataset.git

Cloning into 'Drug-Image-Dataset'...


### Step 3: Reorganize and Balanced Split (80/20 train/val)

In [3]:
import os
import shutil
import random

BINARY_DIR = 'drug_images'
CLONED_DATASET_DIR = 'Drug-Image-Dataset'

# Create directories
for split in ['train', 'val']:
    for cls in ['drug', 'safe']:
        os.makedirs(f'{BINARY_DIR}/{split}/{cls}', exist_ok=True)

# Re-categorize classes so vaping and tobacco are drug (unsafe) matching the local test page
DRUG_FOLDERS = [
    'cannabis', 'cocaine', 'edibles', 'synthetic Cannabinoids',
    'alkylnitrates', 'anorectics', 'coughSyrup', 'vapingProduct', 'tobacco'
]

SAFE_FOLDERS = [
    'antimalarial', 'alcohol'
]

drug_images = []
for folder in DRUG_FOLDERS:
    folder_path = os.path.join(CLONED_DATASET_DIR, folder)
    if os.path.exists(folder_path):
        for fname in os.listdir(folder_path):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                drug_images.append(os.path.join(folder_path, fname))

safe_images = []
for folder in SAFE_FOLDERS:
    folder_path = os.path.join(CLONED_DATASET_DIR, folder)
    if os.path.exists(folder_path):
        for fname in os.listdir(folder_path):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                safe_images.append(os.path.join(folder_path, fname))

print(f'Collected {len(drug_images)} drug images.')
print(f'Collected {len(safe_images)} safe images.')

# Balance classes
random.seed(42)
random.shuffle(drug_images)
random.shuffle(safe_images)

min_count = min(len(drug_images), len(safe_images))
print(f'Balancing dataset to {min_count} samples per class.')
drug_images = drug_images[:min_count]
safe_images = safe_images[:min_count]

split_idx = int(min_count * 0.8)

def copy_files(file_list, split, cls_name):
    dest_dir = os.path.join(BINARY_DIR, split, cls_name)
    for idx, fpath in enumerate(file_list):
        ext = os.path.splitext(fpath)[1]
        dest_path = os.path.join(dest_dir, f'{cls_name}_{idx}{ext}')
        shutil.copy2(fpath, dest_path)

copy_files(drug_images[:split_idx], 'train', 'drug')
copy_files(drug_images[split_idx:], 'val', 'drug')
copy_files(safe_images[:split_idx], 'train', 'safe')
copy_files(safe_images[split_idx:], 'val', 'safe')

print(f'Train Drug: {len(os.listdir(f"{BINARY_DIR}/train/drug"))}')
print(f'Train Safe: {len(os.listdir(f"{BINARY_DIR}/train/safe"))}')
print(f'Val Drug:   {len(os.listdir(f"{BINARY_DIR}/val/drug"))}')
print(f'Val Safe:   {len(os.listdir(f"{BINARY_DIR}/val/safe"))}')

Collected 505 drug images.
Collected 135 safe images.
Balancing dataset to 135 samples per class.
Train Drug: 281
Train Safe: 273
Val Drug:   67
Val Safe:   70


### Step 4: Fine-tune MobileNetV2 with GPU

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder('drug_images/train', transform=train_transform)
val_dataset = datasets.ImageFolder('drug_images/val', transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Freeze base layers first
for param in model.features.parameters():
    param.requires_grad = False

# Unfreeze layer blocks 14 to 18 to allow fine-tuning adaptation
for param in model.features[14:].parameters():
    param.requires_grad = True

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 256),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(256, 2)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

print('Training model classifier...')
best_acc = 0.0
for epoch in range(10):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        
    # Validation
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)
            
    val_acc = val_correct / val_total
    print(f'Epoch {epoch+1}/10 - Loss: {running_loss/len(train_dataset):.4f} - Val Acc: {val_acc*100:.2f}%')

Using device: cpu
Training model classifier...
Epoch 1/10 - Loss: 0.6287 - Val Acc: 78.10%
Epoch 2/10 - Loss: 0.4822 - Val Acc: 84.67%
Epoch 3/10 - Loss: 0.4043 - Val Acc: 83.94%
Epoch 4/10 - Loss: 0.4029 - Val Acc: 81.75%
Epoch 5/10 - Loss: 0.3445 - Val Acc: 82.48%
Epoch 6/10 - Loss: 0.3574 - Val Acc: 83.94%
Epoch 7/10 - Loss: 0.3306 - Val Acc: 81.75%
Epoch 8/10 - Loss: 0.3884 - Val Acc: 81.02%
Epoch 9/10 - Loss: 0.3237 - Val Acc: 82.48%
Epoch 10/10 - Loss: 0.2901 - Val Acc: 78.83%


### Step 5: Export to ONNX format

In [5]:
import json
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(
    model,
    dummy_input,
    'image_model.onnx',
    input_names=['image'],
    output_names=['logits'],
    dynamic_axes={'image': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    opset_version=11
)

classes_mapping = {
    'classes': train_dataset.classes,
    'class_to_idx': train_dataset.class_to_idx
}
with open('image_classes.json', 'w') as f:
    json.dump(classes_mapping, f, indent=2)
print('Saved model and classes mapping successfully!')

C:\Users\asown\AppData\Local\Temp\ipykernel_32996\2660141695.py:4: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0811 13:07:46.772000 32996 Lib\site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


C:\Users\asown\AppData\Local\Programs\Python\Python313\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 11).
Failed to convert the model to the target version 11 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "d:\Project\DrugGuard-Detection-System\venv\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "d:\Project\DrugGuard-Detection-System\venv\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "d:\Project\DrugGuard-Detection-Sys

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
Saved model and classes mapping successfully!


### Step 6: Trigger Download

In [6]:
from google.colab import files
files.download('image_model.onnx')
files.download('image_classes.json')
print('Download initiated!')

ModuleNotFoundError: No module named 'google.colab'